# 🤖 OpenAgentPlatform — Run in Google Colab

**Open-source, local-first AI agent platform that executes tasks, not just chats.**

This notebook lets you run the core engine of [OpenAgentPlatform](https://github.com/iamr1ddl3/open-agent-platform) directly in Google Colab — no desktop install needed.

### What you'll explore:
1. **Environment Setup** — Install Node.js 20 & clone the repo
2. **LLM Gateway** — Connect to OpenAI, Anthropic, Google Gemini, or Ollama
3. **Core Agent Engine** — Create agents with memory and tool execution
4. **40+ Built-in Tools** — File ops, terminal commands, web scraping
5. **Multi-Agent Teams** — Team orchestration with delegation strategies
6. **Scheduler** — Cron-based automation with natural language parsing

> ⚠️ **Note:** The Electron desktop UI won't run in Colab (no display server), but the entire backend engine runs perfectly via Node.js.

---

## 1️⃣ Environment Setup

Install Node.js 20, clone the repo, and install dependencies.

In [ ]:
# Step 1: Install Node.js 20 (Colab ships with an older version)
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - 2>/dev/null
!sudo apt-get install -y nodejs 2>/dev/null | tail -3
!node --version && npm --version

In [ ]:
# Step 2: Clone the OpenAgentPlatform repo
!git clone https://github.com/iamr1ddl3/open-agent-platform.git 2>/dev/null || echo 'Already cloned'
%cd open-agent-platform
!ls -la

In [ ]:
# Step 3: Install npm dependencies (skip Electron postinstall scripts)
!npm install --ignore-scripts 2>&1 | tail -5
print('\n✅ Dependencies installed!')

In [ ]:
# Step 4: Compile TypeScript to JavaScript
!npx tsc --noEmit 2>&1 && echo '✅ TypeScript compilation: PASSED (zero errors)' || echo '❌ Compilation errors found'

In [ ]:
# Step 5: Build the project
!npm run build 2>&1 | tail -10
print('\n✅ Build complete!')

---
## 2️⃣ Configure API Keys

Enter your API keys below. You only need **one** provider to get started.

| Provider | Get a key from | Free tier? |
|----------|---------------|------------|
| OpenAI | [platform.openai.com](https://platform.openai.com) | No |
| Anthropic | [console.anthropic.com](https://console.anthropic.com) | No |
| Google Gemini | [aistudio.google.com](https://aistudio.google.com) | ✅ Yes |
| Ollama | Local only (won't work in Colab) | ✅ Yes |

In [ ]:
# Configure your API keys (only fill in the ones you have)
import os
from getpass import getpass

print("Enter your API keys (press Enter to skip a provider):")
print("="*55)

OPENAI_API_KEY = getpass("🔑 OpenAI API Key: ")
ANTHROPIC_API_KEY = getpass("🔑 Anthropic API Key: ")
GOOGLE_API_KEY = getpass("🔑 Google Gemini API Key: ")

# Set as environment variables for Node.js scripts
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

configured = []
if OPENAI_API_KEY: configured.append('OpenAI')
if ANTHROPIC_API_KEY: configured.append('Anthropic')
if GOOGLE_API_KEY: configured.append('Google Gemini')

if configured:
    print(f"\n✅ Configured providers: {', '.join(configured)}")
else:
    print("\n⚠️ No API keys provided. You can still explore the architecture and tools.")

---
## 3️⃣ LLM Gateway — Multi-Provider Demo

The LLM Gateway provides a **unified interface** to all supported providers. One API, four backends.

In [ ]:
%%writefile /content/open-agent-platform/demo_gateway.js
// Demo: LLM Gateway — Multi-Provider Usage
const { LLMGateway } = require('./dist/llm/gateway');

async function main() {
  const gateway = new LLMGateway();

  // Configure available providers from environment
  const configs = {};

  if (process.env.OPENAI_API_KEY) {
    gateway.configureProvider('openai', {
      apiKey: process.env.OPENAI_API_KEY,
      model: 'gpt-4o-mini',
      maxTokens: 500,
      temperature: 0.7
    });
    configs.openai = true;
    console.log('✅ OpenAI configured (gpt-4o-mini)');
  }

  if (process.env.ANTHROPIC_API_KEY) {
    gateway.configureProvider('anthropic', {
      apiKey: process.env.ANTHROPIC_API_KEY,
      model: 'claude-3-haiku-20240307',
      maxTokens: 500,
      temperature: 0.7
    });
    configs.anthropic = true;
    console.log('✅ Anthropic configured (claude-3-haiku)');
  }

  if (process.env.GOOGLE_API_KEY) {
    gateway.configureProvider('google', {
      apiKey: process.env.GOOGLE_API_KEY,
      model: 'gemini-1.5-flash',
      maxTokens: 500,
      temperature: 0.7
    });
    configs.google = true;
    console.log('✅ Google Gemini configured (gemini-1.5-flash)');
  }

  console.log('\n📋 Available providers:', gateway.listProviders());

  // Test each configured provider
  const prompt = 'Explain what an AI agent is in exactly 2 sentences.';
  console.log(`\n💬 Prompt: "${prompt}"\n`);

  for (const provider of Object.keys(configs)) {
    try {
      console.log(`--- ${provider.toUpperCase()} ---`);
      const response = await gateway.complete({
        messages: [
          { role: 'system', content: 'You are a helpful AI assistant. Be concise.' },
          { role: 'user', content: prompt }
        ],
        maxTokens: 200,
        temperature: 0.7
      }, provider);

      console.log(`Response: ${response.content}`);
      if (response.usage) {
        console.log(`Tokens: ${response.usage.promptTokens} in / ${response.usage.completionTokens} out`);
      }
      console.log();
    } catch (err) {
      console.log(`Error: ${err.message}\n`);
    }
  }

  console.log('🎉 Gateway demo complete!');
}

main().catch(console.error);

In [ ]:
# Run the LLM Gateway demo
!cd /content/open-agent-platform && node demo_gateway.js

---
## 4️⃣ Tool Registry — 40+ Built-in Tools

The ToolRegistry manages all tools the agent can use. Let's register them and explore.

In [ ]:
%%writefile /content/open-agent-platform/demo_tools.js
// Demo: Tool Registry — List and execute tools
const { ToolRegistry } = require('./dist/core/tool-registry');
const { registerBuiltinTools } = require('./dist/tools/index');

async function main() {
  const registry = new ToolRegistry();
  registerBuiltinTools(registry);

  // List all registered tools by category
  const tools = registry.listTools();
  console.log(`\n🔧 Total registered tools: ${tools.length}\n`);

  const categories = {};
  tools.forEach(t => {
    const cat = t.category || 'general';
    if (!categories[cat]) categories[cat] = [];
    categories[cat].push(t.name);
  });

  for (const [cat, names] of Object.entries(categories)) {
    console.log(`📂 ${cat.toUpperCase()} (${names.length} tools)`);
    names.forEach(n => console.log(`   • ${n}`));
    console.log();
  }

  // Demo: Execute file tools
  console.log('--- Execute Tools Demo ---\n');

  // Write a file
  const writeResult = await registry.execute('writeFile', {
    path: '/tmp/oap-test.txt',
    content: 'Hello from OpenAgentPlatform! 🤖\nThis file was created by the agent.'
  });
  console.log('📝 writeFile:', writeResult.success ? 'SUCCESS' : 'FAILED');

  // Read it back
  const readResult = await registry.execute('readFile', {
    path: '/tmp/oap-test.txt'
  });
  console.log('📖 readFile:', readResult.result);

  // Get file info
  const infoResult = await registry.execute('fileInfo', {
    path: '/tmp/oap-test.txt'
  });
  console.log('ℹ️  fileInfo:', JSON.stringify(infoResult.result, null, 2));

  // Execute a terminal command
  const cmdResult = await registry.execute('executeCommand', {
    command: 'echo "Hello from terminal!" && date && whoami'
  });
  console.log('\n💻 executeCommand:', cmdResult.result);

  // List directory
  const dirResult = await registry.execute('listDirectory', {
    path: '/content/open-agent-platform/src'
  });
  console.log('\n📁 listDirectory (src/):', dirResult.result);

  // Show execution statistics
  const stats = registry.getStatistics();
  console.log(`\n📊 Execution Stats:`);
  console.log(`   Total: ${stats.totalExecutions}`);
  console.log(`   Success: ${stats.successfulExecutions}`);
  console.log(`   Failed: ${stats.failedExecutions}`);
  console.log(`   Avg Duration: ${stats.averageDuration.toFixed(1)}ms`);

  console.log('\n🎉 Tool Registry demo complete!');
}

main().catch(console.error);

In [ ]:
# Run the Tool Registry demo
!cd /content/open-agent-platform && node demo_tools.js

---
## 5️⃣ Core Agent Engine — Create & Run an Agent

This is the heart of the platform: an **agentic execution loop** where the agent plans, calls tools, observes results, and iterates.

In [ ]:
%%writefile /content/open-agent-platform/demo_agent.js
// Demo: Core Agent Engine
const { Agent } = require('./dist/core/agent');
const { ToolRegistry } = require('./dist/core/tool-registry');
const { MemoryManager } = require('./dist/core/memory');
const { registerBuiltinTools } = require('./dist/tools/index');

async function main() {
  // ─── Create an Agent ───
  const agent = new Agent({
    id: 'demo-agent-001',
    name: 'Atlas',
    description: 'A versatile AI assistant with file and terminal access',
    systemPrompt: 'You are Atlas, an intelligent AI agent. You can read/write files, run terminal commands, and search the web. Always explain what you are doing before executing a tool.',
    provider: 'openai',
    model: 'gpt-4o-mini',
    temperature: 0.7,
    maxTokens: 2000,
    tools: ['readFile', 'writeFile', 'executeCommand', 'listDirectory'],
    maxIterations: 10
  });

  console.log('🤖 Agent Created:');
  const config = agent.getConfig();
  console.log(`   Name: ${config.name}`);
  console.log(`   Model: ${config.provider}/${config.model}`);
  console.log(`   Tools: ${config.tools.join(', ')}`);
  console.log(`   Max Iterations: ${config.maxIterations}`);

  // ─── Agent Memory ───
  console.log('\n🧠 Memory System:');
  agent.addMemory('long_term', 'User prefers concise responses with code examples', 0.9);
  agent.addMemory('fact', 'The project uses TypeScript with strict mode enabled', 0.8);
  agent.addMemory('short_term', 'Currently exploring the OpenAgentPlatform codebase', 0.5);

  const memories = agent.getMemory();
  memories.forEach(m => {
    console.log(`   [${m.type}] (importance: ${m.importance}) ${m.content}`);
  });

  // ─── Memory Search ───
  console.log('\n🔍 Memory Search ("TypeScript"):');
  const results = agent.searchMemory('TypeScript');
  results.forEach(m => console.log(`   Found: ${m.content}`));

  // ─── Conversation History ───
  console.log('\n💬 Conversation:');
  agent.addMessage('user', 'What files are in the src directory?');
  agent.addMessage('assistant', 'Let me check the src directory for you. I\'ll use the listDirectory tool.');
  agent.addMessage('tool', JSON.stringify({ result: ['core/', 'llm/', 'tools/', 'mcp/', 'teams/'] }));
  agent.addMessage('assistant', 'The src directory contains: core/, llm/, tools/, mcp/, and teams/ — the main modules of the platform.');

  const messages = agent.getAllMessages();
  messages.forEach(m => {
    const preview = m.content.substring(0, 80) + (m.content.length > 80 ? '...' : '');
    console.log(`   [${m.role}] ${preview}`);
  });

  // ─── Formatted Context (what gets sent to LLM) ───
  console.log('\n📋 Formatted Context for LLM:');
  const context = agent.getFormattedContext(5, true);
  console.log(context.substring(0, 500) + '...');

  // ─── Serialization ───
  const serialized = agent.serialize();
  const restored = Agent.deserialize(serialized);
  console.log(`\n💾 Serialization: Agent "${restored.getConfig().name}" restored with ${restored.getAllMessages().length} messages and ${restored.getMemory().length} memories`);

  // ─── Tool Registry Integration ───
  console.log('\n🔧 Tool Registry:');
  const registry = new ToolRegistry();
  registerBuiltinTools(registry);

  // Get tools formatted for LLM function calling
  const llmTools = registry.getToolsForLLM();
  console.log(`   ${llmTools.length} tools available for LLM function calling`);
  console.log(`   Sample tool schema:`);
  console.log(`   ${JSON.stringify(llmTools[0], null, 2).substring(0, 300)}...`);

  // ─── Memory Manager ───
  console.log('\n🧠 Standalone Memory Manager:');
  const memory = new MemoryManager({ maxShortTerm: 50, maxLongTerm: 200 });
  memory.addShortTerm('User asked about API authentication');
  memory.addShortTerm('Discussed OAuth2 flow options');
  memory.addLongTerm('User is building a REST API with Express.js', 0.9);
  memory.addLongTerm('Deployment target is AWS Lambda', 0.8);

  const memStats = memory.getStatistics();
  console.log(`   Total: ${memStats.totalMemories}`);
  console.log(`   Short-term: ${memStats.shortTermCount}`);
  console.log(`   Long-term: ${memStats.longTermCount}`);
  console.log(`   Avg importance: ${memStats.avgImportance.toFixed(2)}`);

  const searchResults = memory.search('API');
  console.log(`\n   Search "API": found ${searchResults.length} memories`);
  searchResults.forEach(m => console.log(`   → ${m.content}`));

  console.log('\n🎉 Agent Engine demo complete!');
}

main().catch(console.error);

In [ ]:
# Run the Agent Engine demo
!cd /content/open-agent-platform && node demo_agent.js

---
## 6️⃣ Full Agent Run — Agentic Execution Loop

This demonstrates the **complete agentic loop**: the agent receives a task, calls tools, observes results, and iterates until the task is done.

> ⚠️ **Requires at least one API key configured above.**

In [ ]:
%%writefile /content/open-agent-platform/demo_agentrun.js
// Demo: Full Agentic Execution Loop
const { Agent } = require('./dist/core/agent');
const { AgentRunner } = require('./dist/core/agent-runner');
const { ToolRegistry } = require('./dist/core/tool-registry');
const { MemoryManager } = require('./dist/core/memory');
const { LLMGateway } = require('./dist/llm/gateway');
const { registerBuiltinTools } = require('./dist/tools/index');

async function main() {
  // Determine which provider to use
  let provider, model, apiKey;
  if (process.env.GOOGLE_API_KEY) {
    provider = 'google'; model = 'gemini-1.5-flash'; apiKey = process.env.GOOGLE_API_KEY;
  } else if (process.env.OPENAI_API_KEY) {
    provider = 'openai'; model = 'gpt-4o-mini'; apiKey = process.env.OPENAI_API_KEY;
  } else if (process.env.ANTHROPIC_API_KEY) {
    provider = 'anthropic'; model = 'claude-3-haiku-20240307'; apiKey = process.env.ANTHROPIC_API_KEY;
  } else {
    console.log('❌ No API key configured. Please run the API key cell first.');
    return;
  }

  console.log(`🔗 Using provider: ${provider} (${model})\n`);

  // Setup
  const gateway = new LLMGateway();
  gateway.configureProvider(provider, { apiKey, model, maxTokens: 1000, temperature: 0.7 });
  gateway.setDefaultProvider(provider);

  const registry = new ToolRegistry();
  registerBuiltinTools(registry);

  const memory = new MemoryManager();

  const agent = new Agent({
    id: 'runner-demo',
    name: 'Atlas',
    description: 'AI agent with file and terminal access',
    systemPrompt: `You are Atlas, a helpful AI agent running in Google Colab. You have access to tools for file operations and terminal commands. When given a task, use your tools to accomplish it. Always explain your reasoning.`,
    provider: provider,
    model: model,
    temperature: 0.7,
    maxTokens: 1000,
    tools: ['readFile', 'writeFile', 'executeCommand', 'listDirectory', 'fileInfo'],
    maxIterations: 5
  });

  // Create the runner
  const runner = new AgentRunner(agent, registry, gateway, memory);

  // Listen to events
  runner.on('start', () => console.log('▶️  Agent started...'));
  runner.on('complete', () => console.log('\n✅ Agent finished!'));
  runner.on('error', (err) => console.log('\n❌ Error:', err.message));

  // Run the agent with a task
  const task = 'Create a file at /tmp/colab-demo.txt with a haiku about AI agents, then read it back and tell me what you wrote.';
  console.log(`📝 Task: "${task}"\n`);

  try {
    const result = await runner.run(task);
    console.log('\n🤖 Agent Response:');
    console.log(result);
  } catch (err) {
    console.error('Agent run failed:', err.message);
  }

  // Show execution log
  const log = registry.getExecutionLog();
  if (log.length > 0) {
    console.log('\n📊 Tool Execution Log:');
    log.forEach((entry, i) => {
      console.log(`   ${i+1}. ${entry.toolName} — ${entry.success ? '✅' : '❌'} (${entry.duration}ms)`);
    });
  }
}

main().catch(console.error);

In [ ]:
# Run the full agentic execution loop
!cd /content/open-agent-platform && node demo_agentrun.js

---
## 7️⃣ Scheduler & Cron Parser

The scheduler supports both **cron expressions** and **natural language** for task scheduling.

In [ ]:
%%writefile /content/open-agent-platform/demo_scheduler.js
// Demo: Scheduler & Cron Parser
const { CronParser } = require('./dist/scheduler/cron-parser');
const { Scheduler } = require('./dist/scheduler/scheduler');

function main() {
  // ─── Cron Parser Demo ───
  console.log('⏰ Cron Parser — Natural Language to Cron\n');

  const expressions = [
    '*/5 * * * *',       // every 5 minutes
    '0 9 * * *',         // daily at 9am
    '0 9 * * 1',         // every Monday at 9am
    '0 */2 * * *',       // every 2 hours
    '0 0 1 * *',         // first of every month
  ];

  expressions.forEach(expr => {
    try {
      const parsed = CronParser.parse(expr);
      const next = CronParser.nextRun(expr);
      console.log(`  "${expr}"`);
      console.log(`    Next run: ${next.toISOString()}`);
      console.log();
    } catch (e) {
      console.log(`  "${expr}" → Error: ${e.message}\n`);
    }
  });

  // ─── Scheduler Demo ───
  console.log('\n📅 Scheduler — Job Management\n');

  const scheduler = new Scheduler();

  // Create scheduled jobs
  const job1 = scheduler.createJob({
    name: 'Daily Report',
    cronExpression: '0 9 * * *',
    agentId: 'reporter-agent',
    task: 'Generate a daily summary report of system metrics',
    enabled: true
  });
  console.log(`  Created: "${job1.name}" (${job1.cronExpression})`);

  const job2 = scheduler.createJob({
    name: 'Weekly Backup',
    cronExpression: '0 2 * * 0',
    agentId: 'backup-agent',
    task: 'Run full database backup and verify integrity',
    enabled: true
  });
  console.log(`  Created: "${job2.name}" (${job2.cronExpression})`);

  const job3 = scheduler.createJob({
    name: 'Hourly Health Check',
    cronExpression: '0 * * * *',
    agentId: 'monitor-agent',
    task: 'Check all API endpoints and report any failures',
    enabled: false
  });
  console.log(`  Created: "${job3.name}" (${job3.cronExpression}) [disabled]`);

  // List all jobs
  console.log(`\n  Total scheduled jobs: ${scheduler.listJobs().length}`);
  scheduler.listJobs().forEach(job => {
    console.log(`    • ${job.name} — ${job.enabled ? '🟢 enabled' : '🔴 disabled'} — agent: ${job.agentId}`);
  });

  // Toggle a job
  scheduler.toggleJob(job3.id, true);
  console.log(`\n  Enabled "${job3.name}"`);

  // Delete a job
  scheduler.deleteJob(job2.id);
  console.log(`  Deleted "${job2.name}"`);

  console.log(`\n  Remaining jobs: ${scheduler.listJobs().length}`);
  scheduler.listJobs().forEach(job => {
    console.log(`    • ${job.name} — ${job.enabled ? '🟢' : '🔴'}`);
  });

  console.log('\n🎉 Scheduler demo complete!');
}

main();

In [ ]:
# Run the Scheduler demo
!cd /content/open-agent-platform && node demo_scheduler.js

---
## 8️⃣ Multi-Agent Teams

Create teams of agents with a **Team Lead** who decomposes tasks and delegates to specialists.

In [ ]:
%%writefile /content/open-agent-platform/demo_teams.js
// Demo: Multi-Agent Teams
const { Agent } = require('./dist/core/agent');
const { ToolRegistry } = require('./dist/core/tool-registry');
const { LLMGateway } = require('./dist/llm/gateway');
const { TeamManager } = require('./dist/teams/team-manager');
const { registerBuiltinTools } = require('./dist/tools/index');

async function main() {
  // Setup
  const gateway = new LLMGateway();
  const registry = new ToolRegistry();
  registerBuiltinTools(registry);

  // ─── Create Specialized Agents ───
  console.log('🤖 Creating specialized agents...\n');

  const agents = [
    new Agent({
      id: 'lead-001',
      name: 'Project Lead',
      description: 'Decomposes complex tasks and coordinates team members',
      systemPrompt: 'You are a project lead who breaks down tasks and delegates to specialists.',
      provider: 'openai', model: 'gpt-4o', temperature: 0.5,
      maxTokens: 2000, tools: [], maxIterations: 5
    }),
    new Agent({
      id: 'researcher-001',
      name: 'Research Analyst',
      description: 'Expert at web research, data gathering, and summarization',
      systemPrompt: 'You are a research analyst who gathers and synthesizes information.',
      provider: 'openai', model: 'gpt-4o-mini', temperature: 0.3,
      maxTokens: 2000, tools: ['webFetch', 'webSearch'], maxIterations: 5
    }),
    new Agent({
      id: 'coder-001',
      name: 'Developer',
      description: 'Writes code, fixes bugs, runs tests',
      systemPrompt: 'You are a senior developer who writes clean, tested code.',
      provider: 'openai', model: 'gpt-4o', temperature: 0.2,
      maxTokens: 3000, tools: ['readFile', 'writeFile', 'executeCommand'], maxIterations: 8
    }),
    new Agent({
      id: 'writer-001',
      name: 'Content Writer',
      description: 'Creates documentation, blog posts, and reports',
      systemPrompt: 'You are an expert technical writer who creates clear documentation.',
      provider: 'openai', model: 'gpt-4o-mini', temperature: 0.7,
      maxTokens: 2000, tools: ['writeFile'], maxIterations: 3
    })
  ];

  agents.forEach(a => {
    const c = a.getConfig();
    console.log(`  🧑 ${c.name} (${c.id})`);
    console.log(`     ${c.description}`);
    console.log(`     Tools: ${c.tools.length > 0 ? c.tools.join(', ') : 'none (coordination only)'}`);
    console.log();
  });

  // ─── Create a Team ───
  const teamManager = new TeamManager(gateway, registry);

  const team = teamManager.createTeam({
    name: 'Product Launch Team',
    description: 'Cross-functional team for product launches',
    lead: 'lead-001',
    agents: ['lead-001', 'researcher-001', 'coder-001', 'writer-001'],
    config: { strategy: 'delegated', maxRounds: 3 }
  });

  console.log('👥 Team Created:');
  console.log(`   Name: ${team.name}`);
  console.log(`   Members: ${team.agents.length}`);
  console.log(`   Lead: ${team.lead}`);
  console.log(`   Strategy: ${team.config?.strategy || 'delegated'}`);

  // List teams
  console.log(`\n📋 All Teams: ${teamManager.listTeams().length}`);
  teamManager.listTeams().forEach(t => {
    console.log(`   • ${t.name} (${t.agents.length} agents)`);
  });

  console.log('\n💡 To run the team with a real task, configure an API key and use:');
  console.log('   await teamManager.runTeam(team.id, "Your task here");');

  console.log('\n🎉 Teams demo complete!');
}

main().catch(console.error);

In [ ]:
# Run the Teams demo
!cd /content/open-agent-platform && node demo_teams.js

---
## 9️⃣ MCP (Model Context Protocol) Client

Connect to external MCP servers to dynamically extend the agent's capabilities.

In [ ]:
%%writefile /content/open-agent-platform/demo_mcp.js
// Demo: MCP Client Architecture
const { MCPClient } = require('./dist/mcp/client');

function main() {
  const client = new MCPClient();

  console.log('🔌 MCP Client — Model Context Protocol\n');
  console.log('The MCP Client supports 3 transport types:\n');
  console.log('  1. stdio   — Spawn a local process, communicate via stdin/stdout (JSON-RPC)');
  console.log('  2. sse     — Connect via Server-Sent Events (streaming)');
  console.log('  3. http    — Polling-based HTTP communication\n');

  console.log('Available methods:');
  console.log('  • client.connect(serverConfig)   — Connect to an MCP server');
  console.log('  • client.disconnect(serverId)     — Disconnect from server');
  console.log('  • client.listServers()            — List connected servers');
  console.log('  • client.listTools(serverId)      — List tools from a server');
  console.log('  • client.executeTool(...)          — Execute a tool on a server\n');

  console.log('Currently connected servers:', client.listServers().length);

  console.log('\nExample — connecting to a filesystem MCP server:');
  console.log(`
  await client.connect({
    id: 'fs-server',
    name: 'Filesystem Server',
    url: 'npx -y @modelcontextprotocol/server-filesystem /tmp',
    transport: 'stdio'
  });
  `);

  console.log('🎉 MCP demo complete!');
}

main();

In [ ]:
# Run the MCP demo
!cd /content/open-agent-platform && node demo_mcp.js

---
## 🧹 Cleanup

Remove demo files and temporary data.

In [ ]:
# Clean up demo files
!cd /content/open-agent-platform && rm -f demo_*.js /tmp/oap-test.txt /tmp/colab-demo.txt
print('✅ Cleanup complete!')

---
## 📚 What's Next?

### Run the full desktop app locally:
```bash
git clone https://github.com/iamr1ddl3/open-agent-platform.git
cd open-agent-platform
npm install
npm run dev
```

### Extend the platform:
- **Create custom skills** — Add `skill.json` manifests to the `skills/` directory
- **Connect MCP servers** — Extend agent capabilities with external tools
- **Build agent teams** — Combine specialists for complex workflows
- **Add channel integrations** — Connect agents to Telegram, Discord, or Slack

### Links:
- 📦 [GitHub Repository](https://github.com/iamr1ddl3/open-agent-platform)
- 📖 [README & Architecture](https://github.com/iamr1ddl3/open-agent-platform/blob/main/README.md)
- 🔗 [Inspired by Accio Work](https://www.accio.com/work/)

---
*Built with TypeScript, React, and Electron. MIT License.*